In [ ]:
#| default_exp work

# Standalone editing and execution

> Atomic cell edits, chapter-scoped execution, and skill installation when working without `aai-coding`.

Inside `aai-coding`, prefer `exhash` and `nbdev-test` for everyday edits and tests. When `nbskill` runs standalone, `edit_nb` and `exec_nb` provide the same safe, cell-addressed loop.

In [ ]:
#| export
import argparse, copy, difflib, sys
from dataclasses import dataclass, field
from importlib.resources import files
from pathlib import Path

from execnb.shell import CaptureShell
from fastcore.nbio import mk_cell, read_nb, validate_nb, write_nb

from nbskill.core import cell_source, chapters, is_eval_false, project_root, source_hash
from nbskill.state import nb_diff_lines

In [ ]:
from tempfile import TemporaryDirectory
from fastcore.nbio import new_nb
from fastcore.test import test_eq, test_fail

## Atomic notebook edits

`edit_nb` validates an entire batch of operations on an in-memory copy before touching disk. Unlike loose patchers, a missing `old` match or a stale `expected_hash` raises immediately so silent partial edits cannot happen.

In [ ]:
#| export
@dataclass
class EditResult:
    "Outcome of one atomic `edit_nb` transaction."
    path: str
    changed: bool
    dry_run: bool
    ops_applied: int
    diff: str

    def __str__(self) -> str:
        mode = "dry-run" if self.dry_run else ("written" if self.changed else "unchanged")
        return f"{self.path}: {mode} ({self.ops_applied} op(s))\n{self.diff}".rstrip()

    def _repr_markdown_(self) -> str: return f"```diff\n{self.diff or '# no changes'}\n```"

def _find_idx(nb, cell_id: str) -> int:
    matches = [i for i, c in enumerate(nb.cells) if str(c.get("id", "")) == cell_id or str(c.get("id", "")).startswith(cell_id)]
    if not matches: raise ValueError(f"Cell ID {cell_id!r} not found")
    if len(matches) > 1: raise ValueError(f"Ambiguous cell ID prefix {cell_id!r}: {[nb.cells[i].id for i in matches]}")
    return matches[0]

def _check_hash(cell, expected_hash: str | None):
    if not expected_hash: return
    actual = source_hash(cell_source(cell), length=len(expected_hash))
    if actual != expected_hash: raise ValueError(f"Stale hash for cell {cell.get('id')}: expected {expected_hash}, got {actual}")

Five operations cover every routine edit: `replace_text`, `replace_cell`, `insert_cell`, `delete_cell`, and `move_cell`. Any modified code cell automatically clears its stale outputs.

In [ ]:
#| export
def _apply_op(nb, op: dict):
    kind = op.get("op")
    if kind == "replace_text":
        idx = _find_idx(nb, op["cell_id"])
        cell = nb.cells[idx]
        _check_hash(cell, op.get("expected_hash"))
        src, old, new = cell_source(cell), op["old"], op["new"]
        count = src.count(old)
        if count == 0: raise ValueError(f"Text {old!r} not found in cell {cell.get('id')}")
        if count > 1 and not op.get("replace_all"): raise ValueError(f"Text {old!r} matched {count} times in cell {cell.get('id')}; pass replace_all=True")
        cell.source = src.replace(old, new) if op.get("replace_all") else src.replace(old, new, 1)
        if cell.get("cell_type") == "code": cell.outputs, cell.execution_count = [], None
    elif kind == "replace_cell":
        idx = _find_idx(nb, op["cell_id"])
        cell = nb.cells[idx]
        _check_hash(cell, op.get("expected_hash"))
        cell.source = op["source"]
        if cell.get("cell_type") == "code": cell.outputs, cell.execution_count = [], None
    elif kind == "insert_cell":
        new_c = mk_cell(op["source"], cell_type=op.get("cell_type", "code"))
        if op.get("before"): nb.cells.insert(_find_idx(nb, op["before"]), new_c)
        elif op.get("after"): nb.cells.insert(_find_idx(nb, op["after"]) + 1, new_c)
        else: nb.cells.append(new_c)
    elif kind == "delete_cell":
        idx = _find_idx(nb, op["cell_id"])
        _check_hash(nb.cells[idx], op.get("expected_hash"))
        nb.cells.pop(idx)
    elif kind == "move_cell":
        idx = _find_idx(nb, op["cell_id"])
        cell = nb.cells.pop(idx)
        if op.get("before"): nb.cells.insert(_find_idx(nb, op["before"]), cell)
        elif op.get("after"): nb.cells.insert(_find_idx(nb, op["after"]) + 1, cell)
        else: nb.cells.append(cell)
    else: raise ValueError(f"Unknown edit op {kind!r}; expected replace_text, replace_cell, insert_cell, delete_cell, or move_cell")

def edit_nb(
    path: str | Path,  # Notebook to modify
    ops: list[dict],  # Ordered list of edit operations
    *,
    dry_run: bool = False,  # Preview diff without writing to disk
) -> EditResult:
    "Apply `ops` atomically with strict match and hash checks, returning a clean cell diff."
    p = Path(path).expanduser()
    orig = read_nb(p)
    draft = copy.deepcopy(orig)
    for op in ops: _apply_op(draft, op)
    validate_nb(draft)
    before, after = nb_diff_lines(orig, include_markdown=True), nb_diff_lines(draft, include_markdown=True)
    diff = "\n".join(difflib.unified_diff(before, after, fromfile=f"before:{p.name}", tofile=f"after:{p.name}", lineterm=""))
    changed = before != after
    if changed and not dry_run: write_nb(draft, p)
    return EditResult(str(p), changed, dry_run, len(ops), diff)

Dry runs leave disk bytes untouched, real writes clear stale outputs, and a missing `old` string fails cleanly instead of succeeding silently.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "edit.ipynb"
    c = mk_cell("answer = 41", outputs=[dict(output_type="stream", name="stdout", text="old\n")])
    write_nb(new_nb([c]), p)
    preview = edit_nb(p, [dict(op="replace_text", cell_id=c.id, old="41", new="42")], dry_run=True)
    test_eq((preview.changed, cell_source(read_nb(p).cells[0])), (True, "answer = 41"))
    applied = edit_nb(p, [dict(op="replace_text", cell_id=c.id, old="41", new="42", expected_hash=source_hash("answer = 41"))])
    test_eq((cell_source(read_nb(p).cells[0]), read_nb(p).cells[0].outputs), ("answer = 42", []))
    test_fail(lambda: edit_nb(p, [dict(op="replace_text", cell_id=c.id, old="missing", new="x")]), contains="not found")
applied

## Focused notebook execution

`exec_nb` executes a notebook from the top through a chosen H2 chapter or cell ID using `execnb.shell.CaptureShell`. It skips `#| eval: false` cells, keeps project imports on `sys.path`, and optionally writes outputs and `executed_hash` fingerprints back to disk.

In [ ]:
#| export
@dataclass
class ExecFailure:
    "Details of the first failing cell during execution."
    cell_id: str
    cell_idx: int
    chapter: str
    ename: str
    evalue: str

@dataclass
class ExecResult:
    "Outcome of executing a notebook scope."
    path: str
    ok: bool
    executed_ids: list[str] = field(default_factory=list)
    skipped_ids: list[str] = field(default_factory=list)
    failure: ExecFailure | None = None
    saved: bool = False

    def __str__(self) -> str:
        if self.ok: return f"{self.path}: ok ({len(self.executed_ids)} executed, {len(self.skipped_ids)} skipped, saved={self.saved})"
        f = self.failure
        return f"{self.path}: FAILED at cell {f.cell_id} [{f.chapter}] — {f.ename}: {f.evalue}"

    def _repr_markdown_(self) -> str: return f"```text\n{self}\n```"

Executing through a chapter runs all preceding setup cells so dependencies stay valid, while leaving later chapters untouched.

In [ ]:
#| export
def _end_index(nb, chapter: str | None, through: str | None) -> int:
    if chapter and through: raise ValueError("Pass at most one of `chapter` or `through`")
    if chapter:
        needle = chapter.strip().lower()
        chaps = chapters(nb)
        matched = [ch for ch in chaps if ch.title.lower() == needle] or [ch for ch in chaps if needle in ch.title.lower()]
        if not matched: raise ValueError(f"Chapter {chapter!r} not found; available: {[c.title for c in chaps]}")
        return matched[0].end
    if through: return _find_idx(nb, through) + 1
    return len(nb.cells)

def exec_nb(
    path: str | Path,  # Notebook to execute
    *,
    chapter: str | None = None,  # Execute from top through the end of this H2 chapter
    through: str | None = None,  # Execute from top through this cell ID (or unique prefix)
    save_outputs: bool = False,  # Persist cell outputs and `executed_hash` metadata
    timeout: int | None = None,  # Optional per-cell timeout in seconds
) -> ExecResult:
    "Execute code cells in `path` through the requested scope and report structured results."
    p = Path(path).expanduser()
    nb = read_nb(p)
    end = _end_index(nb, chapter, through)
    ch_map = {i: ch.title for ch in chapters(nb) for i in range(ch.start, ch.end)}
    root_str = str(project_root(p))
    added_path = root_str not in sys.path
    if added_path: sys.path.insert(0, root_str)
    executed, skipped, failure = [], [], None
    try:
        shell = CaptureShell()
        for idx, cell in enumerate(nb.cells[:end]):
            if cell.get("cell_type") != "code": continue
            cid = str(cell.get("id", f"cell-{idx}"))
            if is_eval_false(cell) or not cell_source(cell).strip():
                skipped.append(cid)
                continue
            outs = shell.run(cell_source(cell), timeout=timeout)
            executed.append(cid)
            if save_outputs:
                cell.outputs = list(outs) if outs else []
                cell.execution_count = len(executed)
                meta = cell.setdefault("metadata", {})
                meta.setdefault("nbskill", {})["executed_hash"] = source_hash(cell_source(cell), length=None)
            if shell.exc:
                failure = ExecFailure(cid, idx, ch_map.get(idx, ""), type(shell.exc).__name__, str(shell.exc))
                break
    finally:
        if added_path and root_str in sys.path: sys.path.remove(root_str)
    if save_outputs: write_nb(nb, p)
    return ExecResult(str(p), failure is None, executed, skipped, failure, save_outputs)

Executing one chapter verifies passing assertions and records `executed_hash` when `save_outputs=True`; failing cells report the exact failing ID and exception.

In [ ]:
with TemporaryDirectory() as tmp:
    p = Path(tmp) / "run.ipynb"
    write_nb(new_nb([mk_cell("## Part 1", "markdown"), mk_cell("x = 21 * 2\nassert x == 42"),
        mk_cell("## Part 2", "markdown"), mk_cell("assert False, 'boom'")]), p)
    ok_res = exec_nb(p, chapter="Part 1", save_outputs=True)
    fail_res = exec_nb(p)
    test_eq((ok_res.ok, len(ok_res.executed_ids), fail_res.ok, fail_res.failure.ename), (True, 1, False, "AssertionError"))
ok_res, fail_res

## Installing `SKILL.md` for external agents

Agents outside `aai-coding` can copy the packaged `SKILL.md` into their skills folder via `install_skill` or the `install_nbskill` CLI.

In [ ]:
#| export
def install_skill(
    skills_dir: str | Path | None = None,  # Destination skills directory (defaults to `~/.codex/skills`)
    *,
    overwrite: bool = False,  # Overwrite an existing differing `SKILL.md`
) -> dict[str, str]:
    "Install `nbskill`'s `SKILL.md` into `<skills_dir>/jupyter-notebooks/SKILL.md`."
    dest_root = Path(skills_dir).expanduser() if skills_dir else Path.home() / ".codex" / "skills"
    target = dest_root / "jupyter-notebooks" / "SKILL.md"
    content = files("nbskill").joinpath("SKILL.md").read_text(encoding="utf-8")
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file():
        if target.read_text(encoding="utf-8") == content: return dict(status="unchanged", path=str(target))
        if not overwrite: raise FileExistsError(f"{target} already exists; pass overwrite=True to replace it")
    target.write_text(content, encoding="utf-8")
    return dict(status="written", path=str(target))

def install_skill_cli(argv: list[str] | None = None) -> int:
    "CLI entry point for `install_nbskill`."
    ap = argparse.ArgumentParser(description="Install the nbskill SKILL.md routing guide.")
    ap.add_argument("--skills-dir", default=None, help="Target skills directory (default: ~/.codex/skills)")
    ap.add_argument("--overwrite", action="store_true", help="Overwrite an existing SKILL.md")
    args = ap.parse_args(argv)
    res = install_skill(args.skills_dir, overwrite=args.overwrite)
    print(f"{res['status']}: {res['path']}")
    return 0

Installing twice into a temporary directory writes the skill once and reports `unchanged` on repeat runs.

In [ ]:
with TemporaryDirectory() as tmp:
    first = install_skill(tmp)
    second = install_skill(tmp)
    test_eq((first["status"], second["status"], install_skill_cli(["--skills-dir", str(tmp)])), ("written", "unchanged", 0))
first